# 02 · Advanced RAG Pipeline
### Practical LLM Engineering — Module 04: RAG Systems

> **What you'll learn**
> - Query transformation: expansion, HyDE, step-back, multi-query
> - Contextual compression and relevance filtering
> - Parent-child chunking and document hierarchy
> - Recursive retrieval and iterative refinement
> - Engineering: pipeline orchestration and tracing

---


## 1. Overview

Basic RAG has a single retrieval step. **Advanced RAG** adds pre-retrieval and post-retrieval stages:

```
PRE-RETRIEVAL                   RETRIEVAL           POST-RETRIEVAL
─────────────                   ─────────           ──────────────
Query transformation            Dense + sparse      Reranking
  • expansion                   Hybrid fusion       Compression
  • HyDE                                            Filtering
  • step-back                                       Citation
  • multi-query                                     extraction

          ↓                         ↓                     ↓
    Transformed             Retrieved chunks         Refined
     queries                (top-k raw)              context
                                                         ↓
                                                       LLM
```


## 2. Setup

In [ ]:
# !pip install numpy matplotlib scikit-learn -q
# ── Shared utilities (carried from Module 03) ────────────────────────
import os, re, json, time, math, random, textwrap
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
from sklearn.preprocessing import normalize

# ── Mock embedder ─────────────────────────────────────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

# ── BM25 ──────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self._corpus, self._doc_freqs, self._avgdl, self._N = [], defaultdict(int), 0.0, 0
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def fit(self, docs):
        self._corpus = [self._tok(d) for d in docs]; self._N = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])
        for doc in self._corpus:
            for t in set(doc): self._doc_freqs[t] += 1
    def _idf(self, t):
        n = self._doc_freqs.get(t, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)
    def get_scores(self, query):
        qt = self._tok(query)
        scores = []
        for doc in self._corpus:
            dl = len(doc); tf = defaultdict(int)
            for t in doc: tf[t] += 1
            s = sum(self._idf(t) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self._avgdl))
                    for t in qt if t in tf)
            scores.append(s)
        return np.array(scores)
    def get_top_k(self, query, k=5):
        s = self.get_scores(query); idx = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in idx]

# ── Vector store ──────────────────────────────────────────────────────
@dataclass
class Doc:
    id: str; text: str; metadata: dict = field(default_factory=dict)

class VectorStore:
    def __init__(self, embedder, dim=64):
        self.embedder = embedder
        self._vecs: np.ndarray = np.zeros((0, dim), dtype=np.float32)
        self._docs: list[Doc]  = []
    def add(self, docs: list[Doc]):
        vecs = self.embedder.embed([d.text for d in docs])
        self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs
        self._docs.extend(docs)
    def search(self, query: str, k=5, where=None) -> list[tuple[Doc, float]]:
        if not self._docs: return []
        q = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        if where:
            for i, doc in enumerate(self._docs):
                for key, val in where.items():
                    if doc.metadata.get(key) != val: scores[i] = -999
        top = np.argsort(-scores)[:k]
        return [(self._docs[i], float(scores[i])) for i in top]
    def __len__(self): return len(self._docs)

# ── Mock LLM ──────────────────────────────────────────────────────────
@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Returns templated answers grounded in provided context."""
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system: str, user: str,
                 max_tokens=512, temperature=0.0) -> LLMResponse:
        time.sleep(0.02)
        ctx_match = re.search(r"Context:(.*?)(?:Question:|$)", user, re.DOTALL)
        ctx = ctx_match.group(1).strip()[:200] if ctx_match else ""
        q   = re.search(r"Question:(.*?)$", user, re.DOTALL)
        q   = q.group(1).strip()[:80] if q else user[:80]
        answer = (f"Based on the provided context, {q.lower().rstrip('?')} "
                  f"can be understood as follows: {ctx[:120]}... "
                  f"This answer is grounded in the retrieved documents.")
        n_in  = len((system+user).split())
        n_out = len(answer.split())
        return LLMResponse(answer, n_in, n_out, 45.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

embedder = MockEmbedder(dim=64)
llm      = MockLLM()
print("✅ Shared utilities loaded (embedder + BM25 + VectorStore + MockLLM)")


In [ ]:
# Build knowledge base (same as NB01)
KNOWLEDGE_BASE = [
    Doc("kb001", "Transformers use self-attention with query-key-value matrices scaled by sqrt(d_k).", {"topic":"ml"}),
    Doc("kb002", "BERT uses masked language modelling and bidirectional attention during pretraining.", {"topic":"ml"}),
    Doc("kb003", "RAG grounds LLM outputs in retrieved documents, reducing hallucination and adding citations.", {"topic":"rag"}),
    Doc("kb004", "Vector databases index dense embeddings for ANN search using HNSW or IVF algorithms.", {"topic":"retrieval"}),
    Doc("kb005", "BM25 ranks documents by TF-IDF with length normalisation; standard in Elasticsearch.", {"topic":"retrieval"}),
    Doc("kb006", "Chunk size determines embedding granularity; 256-512 tokens balances quality and precision.", {"topic":"rag"}),
    Doc("kb007", "Hybrid search fuses dense and sparse retrieval scores for better recall across query types.", {"topic":"retrieval"}),
    Doc("kb008", "LoRA fine-tunes LLMs by adding low-rank matrices to attention weights, reducing parameters.", {"topic":"ml"}),
    Doc("kb009", "Chain-of-thought prompting improves reasoning by asking the model to show its work.", {"topic":"prompting"}),
    Doc("kb010", "KV cache avoids re-computing attention keys and values for past tokens during generation.", {"topic":"inference"}),
    Doc("kb011", "RRF merges ranked retrieval lists by assigning score 1/(60+rank) per list position.", {"topic":"retrieval"}),
    Doc("kb012", "Cross-encoders score query-document pairs jointly for higher-accuracy reranking.", {"topic":"retrieval"}),
    Doc("kb013", "Parent-child chunking stores small retrieval chunks linked to larger parent context chunks.", {"topic":"rag"}),
    Doc("kb014", "HyDE generates a hypothetical document for the query, then retrieves using its embedding.", {"topic":"rag"}),
    Doc("kb015", "Multi-query retrieval generates query variants to improve recall through diverse retrievals.", {"topic":"rag"}),
]
store = VectorStore(embedder)
store.add(KNOWLEDGE_BASE)
bm25 = BM25(); bm25.fit([d.text for d in KNOWLEDGE_BASE])
print(f"KB: {len(store)} docs | BM25 fitted")


## 3. Pre-Retrieval: Query Transformation

### 3.1 Why raw queries often fail

User queries are often:
- Too short ("RAG chunking") — little embedding signal
- Ambiguous ("how does it work") — unclear reference
- Wrong vocabulary ("fast search") — embedding misses "ANN", "FAISS"

Query transformation rewrites or expands the query before retrieval.


In [ ]:
class QueryTransformer:
    """Collection of query transformation strategies."""

    def __init__(self, llm: MockLLM, embedder: MockEmbedder):
        self.llm      = llm
        self.embedder = embedder

    # ── Strategy 1: Query expansion ───────────────────────────────────
    def expand(self, query: str, n_terms: int = 5) -> str:
        """Add related terms to the query to improve recall."""
        system = "You are a search query expansion assistant."
        user   = (f"Expand this search query with {n_terms} related technical terms. "
                  f"Return the expanded query as a single line.\n\nQuery: {query}")
        resp   = self.llm(system, user, max_tokens=80)
        # Combine original + expansion
        return f"{query} {resp.text.strip()}"

    # ── Strategy 2: HyDE (Hypothetical Document Embeddings) ──────────
    def hyde(self, query: str) -> str:
        """Generate a hypothetical answer, use it as the retrieval query."""
        system = "You are a technical knowledge base. Write a brief factual answer."
        user   = f"Write a 2-sentence factual answer to: {query}"
        resp   = self.llm(system, user, max_tokens=100)
        return resp.text.strip()

    # ── Strategy 3: Step-back ─────────────────────────────────────────
    def step_back(self, query: str) -> str:
        """Generate a broader, more general version of the query."""
        system = "You are a search assistant that generalises queries."
        user   = f"Write a broader, more general version of: {query}"
        resp   = self.llm(system, user, max_tokens=50)
        return resp.text.strip()

    # ── Strategy 4: Multi-query ───────────────────────────────────────
    def multi_query(self, query: str, n: int = 3) -> list[str]:
        """Generate n alternative phrasings of the query."""
        system = "You are a search assistant that rephrases queries."
        user   = (f"Generate {n} alternative phrasings of the following query. "
                  f"Output one per line.\n\nQuery: {query}")
        resp   = self.llm(system, user, max_tokens=150)
        variants = [q.strip() for q in resp.text.strip().split("\n") if q.strip()]
        return [query] + variants[:n]


qt = QueryTransformer(llm, embedder)
original = "how does retrieval work with dense vectors"

print(f"Original : {original}")
print(f"Expanded : {qt.expand(original)[:80]}")
print(f"HyDE     : {qt.hyde(original)[:80]}")
print(f"Step-back: {qt.step_back(original)[:80]}")
print(f"Multi-Q  : {qt.multi_query(original, 3)}")


## 4. Multi-Query Retrieval and Deduplication

In [ ]:
def multi_query_retrieve(query: str, store: VectorStore,
                          qt: QueryTransformer, k_per_query: int = 3,
                          n_variants: int = 2) -> list[tuple[Doc, float]]:
    """
    Retrieve using multiple query variants; deduplicate and re-rank by best score.
    """
    variants = qt.multi_query(query, n_variants)
    seen:  dict[str, float] = {}
    order: dict[str, Doc]   = {}

    for variant in variants:
        for doc, score in store.search(variant, k=k_per_query):
            if doc.id not in seen or score > seen[doc.id]:
                seen[doc.id]  = score
                order[doc.id] = doc

    # Sort by best score across all variants
    sorted_ids = sorted(seen.keys(), key=lambda x: -seen[x])
    return [(order[i], seen[i]) for i in sorted_ids]


# Compare single-query vs multi-query recall
queries = [
    "ANN vector search",
    "LLM grounding retrieval",
    "transformer architecture",
]

print(f"{'Query':<35} {'Single-Q top-3':^30} {'Multi-Q top-3':^30}")
print("-" * 96)
for q in queries:
    single = [doc.id for doc, _ in store.search(q, k=3)]
    multi  = [doc.id for doc, _ in multi_query_retrieve(q, store, qt, k_per_query=3, n_variants=2)[:3]]
    print(f"{q:<35} {str(single):<30} {str(multi):<30}")


## 5. Post-Retrieval: Contextual Compression

In [ ]:
def compress_context(doc: Doc, query: str, llm: MockLLM,
                      max_tokens: int = 60) -> Optional[str]:
    """
    Extract only the query-relevant sentences from a retrieved chunk.
    Returns None if the chunk has no relevant content.
    """
    system = "You are a context extraction assistant."
    user   = (f"Extract only the sentences relevant to the following question. "
              f"If no sentence is relevant, reply 'IRRELEVANT'.\n\n"
              f"Question: {query}\n\nDocument: {doc.text}")
    resp   = llm(system, user, max_tokens=max_tokens)
    result = resp.text.strip()
    return None if "IRRELEVANT" in result.upper() else result


def relevance_filter(retrieved: list[tuple[Doc, float]],
                      min_score: float = 0.25) -> list[tuple[Doc, float]]:
    """Remove chunks below a minimum relevance threshold."""
    return [(doc, score) for doc, score in retrieved if score >= min_score]


# Demo
query  = "how does approximate nearest neighbour search work"
raw    = store.search(query, k=5)
filtered = relevance_filter(raw, min_score=0.20)

print(f"Before filtering: {len(raw)} chunks")
print(f"After  filtering: {len(filtered)} chunks (min_score=0.20)\n")

print("Contextual compression:")
for doc, score in filtered[:3]:
    compressed = compress_context(doc, query, llm)
    status = "✅" if compressed else "❌ (filtered)"
    print(f"  {status} [{doc.id}] score={score:.3f}")
    if compressed:
        print(f"     → {compressed[:80]}...")


## 6. Parent-Child Chunking

In [ ]:
@dataclass
class ParentDoc:
    id:   str
    text: str

@dataclass
class ChildChunk:
    id:        str
    parent_id: str
    text:      str
    metadata:  dict = field(default_factory=dict)


class ParentChildStore:
    """
    Retrieval indexes child chunks (small, precise embeddings).
    Generation uses parent documents (full context).
    """
    def __init__(self, embedder):
        self.embedder  = embedder
        self._parents: dict[str, ParentDoc]  = {}
        self._children: list[ChildChunk]     = []
        self._child_vecs: Optional[np.ndarray] = None

    def add_document(self, doc_id: str, text: str, chunk_size: int = 100):
        """Split document into child chunks; store parent text."""
        parent = ParentDoc(id=doc_id, text=text)
        self._parents[doc_id] = parent

        # Simple word-based chunking
        words  = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size):
            chunk_text = " ".join(words[i:i+chunk_size])
            chunk_id   = f"{doc_id}_chunk_{i//chunk_size}"
            chunks.append(ChildChunk(id=chunk_id, parent_id=doc_id, text=chunk_text))

        self._children.extend(chunks)
        # Rebuild child vector matrix
        all_child_vecs = self.embedder.embed([c.text for c in self._children])
        self._child_vecs = all_child_vecs

    def search(self, query: str, k: int = 3) -> list[tuple[ParentDoc, float, str]]:
        """Retrieve by child similarity; return parent documents."""
        if not self._children: return []
        q_vec  = self.embedder.embed([query])[0]
        scores = self._child_vecs @ q_vec
        top_k  = np.argsort(-scores)[:k]

        # Deduplicate by parent_id (keep best child score per parent)
        seen: dict[str, tuple[ParentDoc, float, str]] = {}
        for i in top_k:
            child  = self._children[i]
            parent = self._parents[child.parent_id]
            score  = float(scores[i])
            if child.parent_id not in seen or score > seen[child.parent_id][1]:
                seen[child.parent_id] = (parent, score, child.id)

        return sorted(seen.values(), key=lambda x: -x[1])


# Demo with longer documents
LONG_DOCS = [
    ("doc_A",
     "Transformers are the dominant architecture in modern NLP. "
     "They use multi-head self-attention to model long-range dependencies. "
     "The attention mechanism computes query-key dot products scaled by sqrt(d_k). "
     "BERT uses bidirectional attention and is pre-trained with masked language modelling. "
     "GPT uses causal (unidirectional) attention and is pre-trained autoregressively."),
    ("doc_B",
     "RAG improves LLM factuality by grounding responses in retrieved documents. "
     "The pipeline has two phases: offline indexing and online retrieval-generation. "
     "Chunking splits documents before embedding to improve retrieval precision. "
     "Hybrid search combines dense and sparse retrieval for best recall. "
     "Reranking with a cross-encoder further improves result relevance."),
]

pc_store = ParentChildStore(embedder)
for doc_id, text in LONG_DOCS:
    pc_store.add_document(doc_id, text, chunk_size=15)

print(f"Parent-child store: {len(pc_store._parents)} parents, {len(pc_store._children)} children\n")
results_pc = pc_store.search("how does BERT use attention", k=2)
for parent, score, child_id in results_pc:
    print(f"Matched child: {child_id} (score={score:.3f})")
    print(f"Returned parent [{parent.id}]: {parent.text[:120]}...")
    print()


## 7. Advanced RAG Pipeline (Orchestrated)

In [ ]:
class AdvancedRAGPipeline:
    """
    Advanced RAG with: multi-query → hybrid retrieve → filter → compress → generate.
    Includes pipeline tracing for observability.
    """
    def __init__(self, vector_store, bm25_index, llm, embedder,
                 k=5, min_score=0.20, use_compression=True,
                 use_multi_query=True):
        self.store       = vector_store
        self.bm25        = bm25_index
        self.llm         = llm
        self.embedder    = embedder
        self.k           = k
        self.min_score   = min_score
        self.compress    = use_compression
        self.multi_q     = use_multi_query
        self.qt          = QueryTransformer(llm, embedder)

    def _hybrid_retrieve(self, query: str) -> list[tuple[Doc, float]]:
        dense_docs  = {doc.id: (doc, sc) for doc, sc in self.store.search(query, k=self.k)}
        sparse_ids  = [i for i, _ in self.bm25.get_top_k(query, k=self.k)]
        sparse_docs = {KNOWLEDGE_BASE[i].id: (KNOWLEDGE_BASE[i],
                        self.bm25.get_scores(query)[i])
                       for i in sparse_ids}

        all_ids = set(dense_docs) | set(sparse_docs)
        combined = {}
        for doc_id in all_ids:
            d_sc = dense_docs[doc_id][1] if doc_id in dense_docs else 0.0
            s_sc = sparse_docs[doc_id][1] if doc_id in sparse_docs else 0.0
            # Normalise sparse to [0,1] range heuristically
            s_sc_norm = min(s_sc / 3.0, 1.0)
            combined[doc_id] = (dense_docs.get(doc_id, sparse_docs[doc_id])[0],
                                 0.6 * d_sc + 0.4 * s_sc_norm)
        return sorted(combined.values(), key=lambda x: -x[1])

    def query(self, question: str) -> dict:
        trace = {"stages": {}}
        t_total = time.perf_counter()

        # Stage 1: Query transformation
        t0 = time.perf_counter()
        if self.multi_q:
            variants = self.qt.multi_query(question, n=2)
        else:
            variants = [question]
        trace["stages"]["query_transform"] = {
            "ms": (time.perf_counter()-t0)*1000,
            "variants": variants
        }

        # Stage 2: Hybrid retrieval over all variants
        t0 = time.perf_counter()
        seen: dict[str, tuple[Doc, float]] = {}
        for v in variants:
            for doc, sc in self._hybrid_retrieve(v):
                if doc.id not in seen or sc > seen[doc.id][1]:
                    seen[doc.id] = (doc, sc)
        raw = sorted(seen.values(), key=lambda x: -x[1])[:self.k]
        trace["stages"]["retrieval"] = {
            "ms": (time.perf_counter()-t0)*1000,
            "n_raw": len(raw),
            "top_scores": [round(sc, 3) for _, sc in raw[:3]]
        }

        # Stage 3: Relevance filtering
        filtered = relevance_filter(raw, self.min_score)
        trace["stages"]["filter"] = {"n_after": len(filtered)}

        # Stage 4: Contextual compression
        t0 = time.perf_counter()
        if self.compress:
            context_parts = []
            for doc, sc in filtered[:4]:
                compressed = compress_context(doc, question, self.llm)
                if compressed:
                    context_parts.append(f"[{doc.id}] {compressed}")
        else:
            context_parts = [f"[{doc.id}] {doc.text}" for doc, _ in filtered[:4]]
        trace["stages"]["compression"] = {
            "ms": (time.perf_counter()-t0)*1000,
            "n_chunks": len(context_parts)
        }

        # Stage 5: Generation
        t0     = time.perf_counter()
        ctx    = "\n".join(context_parts)
        system = "You are an expert assistant. Answer using only the provided context. Cite sources."
        user   = f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:"
        resp   = self.llm(system, user, max_tokens=300)
        trace["stages"]["generation"] = {"ms": (time.perf_counter()-t0)*1000}

        trace["total_ms"] = (time.perf_counter() - t_total) * 1000

        return {
            "question": question,
            "answer":   resp.text,
            "sources":  [doc.id for doc, _ in filtered[:4]],
            "trace":    trace,
        }


adv_rag = AdvancedRAGPipeline(store, bm25, llm, embedder,
                               k=5, min_score=0.15, use_compression=True,
                               use_multi_query=True)

print("Advanced RAG Pipeline\n")
for q in ["How does hybrid search improve retrieval?",
           "What is HyDE and when should I use it?"]:
    r = adv_rag.query(q)
    print(f"Q: {q}")
    print(f"A: {r['answer'][:120]}...")
    print(f"Sources: {r['sources']}")
    stages = r['trace']['stages']
    print(f"Latency: " + "  ".join(f"{k}={v['ms']:.0f}ms"
                                    for k, v in stages.items() if 'ms' in v))
    print()


## 8. Key Takeaways

| Concept | Summary |
|---|---|
| **Query expansion** | Add related terms to improve dense retrieval recall |
| **HyDE** | Generate a hypothetical answer → embed it → retrieve with it |
| **Multi-query** | Generate variants → retrieve all → deduplicate → best scores |
| **Contextual compression** | Extract only query-relevant sentences per chunk |
| **Relevance filtering** | Drop chunks below a minimum score threshold |
| **Parent-child** | Retrieve small chunks; return large parent context to LLM |
| **Pipeline tracing** | Instrument each stage for latency and quality observability |

---
**Next notebook →** `03_reranking.ipynb`
